

```python
self.proxy_model.setFilterRole(Qt.ItemDataRole.DisplayRole)
```

---

### 一、什么是 ItemDataRole？

在 Qt Model/View 架构中，**每个单元格可以同时存储多份数据**，通过不同的"角色 (Role)"来区分：

```python
item = QStandardItem("张三")                    # DisplayRole = "张三"
item.setData(25, Qt.ItemDataRole.UserRole)      # UserRole    = 25 (年龄)
item.setData("zhangsan", Qt.ItemDataRole.UserRole + 1)  # 自定义数据
```

可以理解为每个 `QStandardItem` 内部是一个字典：

```
item = {
    DisplayRole:  "张三",     # 显示给用户看的文本
    EditRole:     "张三",     # 编辑时的文本
    ToolTipRole:  None,       # 鼠标悬停提示
    UserRole:     25,         # 自定义数据
    ...
}
```

常用 Role：

| Role | 用途 |
|---|---|
| `DisplayRole` | 视图中显示的文本 |
| `EditRole` | 编辑器中编辑的文本 |
| `ToolTipRole` | 鼠标悬停提示 |
| `UserRole` | 自定义数据（程序员自由使用） |
| `DecorationRole` | 图标 |
| `CheckStateRole` | 复选框状态 |

---

### 二、`setFilterRole` 做什么？

它告诉 `QSortFilterProxyModel`：**用哪个 Role 的数据来匹配过滤条件。**

```python
# 用户输入 "技术" 触发过滤
# self.proxy_model.setFilterRegularExpression("技术")  ← 第 157 行

# 代理模型开始逐行扫描，对每个 item：
#   → 取出 item.data(DisplayRole)  即 "技术部"
#   → 用正则匹配 "技术" in "技术部" → 匹配！保留该行
#   → 取出 item.data(DisplayRole)  即 "市场部"
#   → 用正则匹配 "技术" in "市场部" → 不匹配！隐藏该行
```

流程图：

```
用户输入 "技术"
       ↓
setFilterRegularExpression("技术")
       ↓
QSortFilterProxyModel 遍历每一行
       ↓
取出每个 item 的 DisplayRole 数据  ← 由 setFilterRole 决定取哪个 Role
       ↓
用正则表达式匹配
       ↓
匹配的行 → 显示 / 不匹配的行 → 隐藏
```

---

### 三、为什么选 `DisplayRole` 而不是其他？

在这段代码中（[第 127-128 行](file:///Users/usst_ziyi/Programs/solo/pyqt6-1/units/unit6_modelview_qss/01_modelview.py#L127-L128)）：

```python
row_items = [QStandardItem(d) for d in data]
```

每个 `QStandardItem(d)` 创建时，传入的 `d` 字符串（如 `"张三"`、`"技术部"`）自动存入 `DisplayRole`。所以 `DisplayRole` 就是**用户看到的文本**，用这个来搜索最自然。

### 举个例子说明不同 Role 的效果

假设表中显示的是部门名称（`DisplayRole`），但内部存了部门编号（`UserRole`）：

```python
item = QStandardItem()
item.setData("技术部",     Qt.ItemDataRole.DisplayRole)  # 显示："技术部"
item.setData("DEPT_001",  Qt.ItemDataRole.UserRole)      # 隐藏：部门编号
```

| 设置 | 用户输入 "技术" | 用户输入 "DEPT" |
|---|---|---|
| `.setFilterRole(DisplayRole)` | ✅ 匹配 "技术部" | ❌ 不匹配 |
| `.setFilterRole(UserRole)` | ❌ 不匹配 "DEPT_001" | ✅ 匹配 "DEPT_001" |

```python
# 场景：用隐藏的部门编号搜索
self.proxy_model.setFilterRole(Qt.ItemDataRole.UserRole)
self.proxy_model.setFilterRegularExpression("DEPT_001")  # 搜索编号，而非显示文本
```

---

### 四、总结

这行代码就是设定：**当用户在搜索框输入文字时，代理模型去匹配每个单元格"显示出来"的文本 (`DisplayRole`)**，而不是隐藏数据或其他 Role。这基本是 `QSortFilterProxyModel` 的默认行为，但显式写出来让意图更清晰。